In [1]:
!pip install transformers[torch] datasets evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00


In [2]:
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import evaluate
import numpy as np
from google.colab import drive
import pandas as pd

In [3]:
drive.mount('/content/drive')

train_data = '/content/drive/MyDrive/Colab Notebooks/GermanDistilBERT/zwischenfragen_annotations.csv'

df = pd.read_csv(train_data, sep=";")
print(df.head())

Mounted at /content/drive
  document_id                                      zwischenfrage  annotation
0   BT_10_004  Nein. — Diese intellektuelle Unredlichkeit, He...           0
1   BT_10_004  Nein, ich halte es wie mein Vorgänger. Die Sow...           0
2   BT_10_004                                              Nein.           0
3   BT_10_004  Nein, ich möchte es angesichts der begrenzten ...           0
4   BT_10_004  Nein, ich gestatte keine Zwischenfrage. Da ich...           0


In [4]:
# Convert list of tuples to a Hugging Face Dataset
texts = df.zwischenfrage
labels = df.annotation

raw_dataset = Dataset.from_dict({"text": texts, "label": labels})
print(raw_dataset)

# Split into Train and Test (80/20)
dataset_dict = raw_dataset.train_test_split(test_size=0.2)

Dataset({
    features: ['text', 'label'],
    num_rows: 166
})


In [ ]:
model_name = "distilbert/distilbert-base-german-cased"
num_labels = 2

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding=True)

tokenized_datasets = dataset_dict.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/464 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [ ]:
id2label = {0: "NEGATIVE", 1: "POSITIVE"} # Change these to your actual category names
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)
for param in model.distilbert.parameters():
    param.requires_grad = False

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-german-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
metric = evaluate.load("f1") # F1-score is often better for binary tasks

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
training_args = TrainingArguments(
    output_dir="./german-binary-classifier",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=4, # Slightly more epochs for smaller custom datasets
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    #tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()
trainer.save_model("./my_custom_german_model")

Epoch,Training Loss,Validation Loss,F1
1,No log,0.629393,0.914286
2,No log,0.606685,0.914286
3,No log,0.594286,0.914286
4,No log,0.590743,0.914286


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# Inference Test
from transformers import pipeline

pipe = pipeline(
    "text-classification",
    model="./my_custom_german_model",
    tokenizer="./my_custom_german_model"
)

# 2. Predict on a single sentence
text = "Ich nehme die Zwischenfrage gerne an."
result = pipe(text)
print(f"Result: {result}")

# 3. Predict on a list of sentences
sequences = [
    "Dafür habe ich nun leider keine Zeit.",
    "Noch nicht.",
    "Beim nächsten Mal.",
    "Ja, sehr gerne Frau Kollegin."
]
results = pipe(sequences)

for text, res in zip(sequences, results):
    print(f"\nText: {text}")
    print(f"Label: {res['label']}, Score: {res['score']:.4f}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Result: [{'label': 'POSITIVE', 'score': 0.5221725702285767}]

Text: Dafür habe ich nun leider keine Zeit.
Label: POSITIVE, Score: 0.5459

Text: Noch nicht.
Label: POSITIVE, Score: 0.5749

Text: Beim nächsten Mal.
Label: POSITIVE, Score: 0.6082

Text: Ja, sehr gerne Frau Kollegin.
Label: POSITIVE, Score: 0.5588


## TF-IDF approach

In [5]:
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, accuracy_score


# 2. Split the data
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

# 3. Create the Pipeline
# We use German stop words because common words like "der, die, das"
# carry no meaning for classification.
text_clf = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),   # Look at single words AND pairs (e.g., "nicht gut")
        max_features=1000,    # Limit features so we don't overfit
        stop_words=None       # We'll handle German stops below or keep them
    )),
    ('clf', LogisticRegression(C=1.0)), # C is regularization; lower C = less overfitting
])

# 4. Train
text_clf.fit(X_train, y_train)

# 5. Evaluate
predictions = text_clf.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, predictions):.2%}")
print(classification_report(y_test, predictions))

# 6. Save the model
joblib.dump(text_clf, 'german_tfidf_model.pkl')

Accuracy: 85.29%
              precision    recall  f1-score   support

           0       0.81      0.87      0.84        15
           1       0.89      0.84      0.86        19

    accuracy                           0.85        34
   macro avg       0.85      0.85      0.85        34
weighted avg       0.86      0.85      0.85        34



['german_tfidf_model.pkl']

In [6]:
# For Inference
import joblib

# Load the model
model = joblib.load('german_tfidf_model.pkl')

# Predict
new_texts = [
    "Dafür habe ich nun leider keine Zeit.",
    "Noch nicht.",
    "Nein, Beim nächsten Mal.",
    "Ja, oder nein, nein!.",
    "Bitte, Herr Kollege"
]
preds = model.predict(new_texts)
probs = model.predict_proba(new_texts) # Returns probability for [Class 0, Class 1]

for text, p, prob in zip(new_texts, preds, probs):
    print(f"Text: {text} | Label: {p} | Confidence: {max(prob):.2%}")

Text: Dafür habe ich nun leider keine Zeit. | Label: 0 | Confidence: 58.17%
Text: Noch nicht. | Label: 0 | Confidence: 57.92%
Text: Nein, Beim nächsten Mal. | Label: 0 | Confidence: 63.16%
Text: Ja, oder nein, nein!. | Label: 0 | Confidence: 63.53%
Text: Bitte, Herr Kollege | Label: 1 | Confidence: 73.17%
